In [1]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import numpy as np

In [4]:
ratings = pd.read_csv(
    "/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)

movies_ml = pd.read_csv(
    "/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/movies.dat",
    sep="::",
    engine="python",
    names=["movieId", "title", "genres"],
    encoding="latin-1"
)

In [5]:
movies_ml.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
movies_meta = pd.read_csv(
    "/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/movies_metadata.xls",
    low_memory=False
)
# Keep only valid TMDB ids
movies_meta = movies_meta[movies_meta["id"].str.isnumeric()]
movies_meta["id"] = movies_meta["id"].astype(int)


In [7]:
credits = pd.read_csv('/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/credits.xls')
keywords = pd.read_csv('/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/keywords.xls')
tmdb = movies_meta.merge(credits, on="id") \
                   .merge(keywords, on="id")


In [8]:

tmdb.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count', 'cast', 'crew', 'keywords'],
      dtype='object')

In [9]:
tmdb["homepage"]

,homepage
0,http://toystory.disney.com/toy-story
1,NaN
2,NaN
3,NaN
4,NaN
...,...
46623,http://www.imdb.com/title/tt6209470/
46624,NaN
46625,NaN
46626,NaN


In [10]:
tmdb["poster_path"]

,poster_path
0,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg
1,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg
2,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg
3,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg
4,/e64sOI48hQXyru7naBFyssKFxVd.jpg
...,...
46623,/jldsYflnId4tTWPx8es3uzsB1I8.jpg
46624,/xZkmxsNmYXJbKVsTRLLx3pqGHx7.jpg
46625,/d5bX92nDsISNhu3ZT69uHwmfCGw.jpg
46626,/aorBPO7ak8e8iJKT5OcqYxU3jlK.jpg


In [11]:
links = pd.read_csv(
    "/content/drive/MyDrive/HYBRID RECOMMENDER SYSTEM/links.xls"
)
# Clean ids
links = links.dropna(subset=["tmdbId", "movieId"])
links["tmdbId"] = links["tmdbId"].astype(int)


In [12]:
movies_common = movies_ml.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="inner"
).merge(
    tmdb,
    left_on="tmdbId",
    right_on="id",
    how="inner"
)


In [13]:
print("MovieLens movies:", movies_ml.movieId.nunique())
print("Common movies:", movies_common.movieId.nunique())


MovieLens movies: 3883
Common movies: 3828


In [14]:
ratings.shape

(1000209, 4)

In [15]:
common_movie_ids = set(movies_common.movieId)

ratings_clean = ratings[
    ratings.movieId.isin(common_movie_ids)
]


In [16]:
ratings.shape

(1000209, 4)

In [17]:
ratings_clean.shape


(997407, 4)

In [18]:
ratings_clean

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [19]:
movies_common.isnull().sum()

,0
movieId,0
title_x,0
genres_x,0
tmdbId,0
adult,0
belongs_to_collection,3194
budget,0
genres_y,0
homepage,3623
id,0


In [20]:
content_df = movies_common[[
    "movieId",
    "title_x",
    "overview",
    "genres_x",
    "keywords",
    "cast",
    "crew"
]].copy()


In [21]:
content_df.isnull().sum()

,0
movieId,0
title_x,0
overview,18
genres_x,0
keywords,0
cast,0
crew,0


In [22]:
content_df.rename(columns={"title_x":"title","genres_x":"genres"},inplace=True)

In [23]:
content_df.dropna(inplace=True)

In [24]:
content_df

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation|Children's|Comedy,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure|Children's|Fantasy,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy|Romance,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...","[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy|Drama,"[{'id': 818, 'name': 'based on novel'}, {'id':...","[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,"[{'id': 591, 'name': 'cia'}, {'id': 822, 'name...","[{'cast_id': 1, 'character': 'Gaylord ""Greg"" F...","[{'credit_id': '52fe4303c3a36847f8033ca9', 'de..."
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,"[{'id': 1803, 'name': 'drug addiction'}, {'id'...","[{'cast_id': 29, 'character': 'Sara Goldfarb',...","[{'credit_id': '52fe4263c3a36847f801a72b', 'de..."
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,"[{'id': 10183, 'name': 'independent film'}, {'...","[{'cast_id': 1, 'character': 'Pvt. Roland Bozz...","[{'credit_id': '52fe43a39251416c75018317', 'de..."
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,[],"[{'cast_id': 1, 'character': 'Buddy Visalo', '...","[{'credit_id': '52fe46c2c3a368484e0a2471', 'de..."


In [25]:
content_df["genres"]=content_df["genres"].str.replace("|"," ")

In [26]:
content_df

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...","[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,"[{'id': 818, 'name': 'based on novel'}, {'id':...","[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,"[{'id': 591, 'name': 'cia'}, {'id': 822, 'name...","[{'cast_id': 1, 'character': 'Gaylord ""Greg"" F...","[{'credit_id': '52fe4303c3a36847f8033ca9', 'de..."
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,"[{'id': 1803, 'name': 'drug addiction'}, {'id'...","[{'cast_id': 29, 'character': 'Sara Goldfarb',...","[{'credit_id': '52fe4263c3a36847f801a72b', 'de..."
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,"[{'id': 10183, 'name': 'independent film'}, {'...","[{'cast_id': 1, 'character': 'Pvt. Roland Bozz...","[{'credit_id': '52fe43a39251416c75018317', 'de..."
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,[],"[{'cast_id': 1, 'character': 'Buddy Visalo', '...","[{'credit_id': '52fe46c2c3a368484e0a2471', 'de..."


In [27]:
import ast


In [28]:
def extract_names(text):
    data=ast.literal_eval(text)
    return " ".join([d["name"] for d in data])


In [29]:
content_df["keywords"][0]

"[{'id': 931, 'name': 'jealousy'}, {'id': 4290, 'name': 'toy'}, {'id': 5202, 'name': 'boy'}, {'id': 6054, 'name': 'friendship'}, {'id': 9713, 'name': 'friends'}, {'id': 9823, 'name': 'rivalry'}, {'id': 165503, 'name': 'boy next door'}, {'id': 170722, 'name': 'new toy'}, {'id': 187065, 'name': 'toy comes to life'}]"

In [30]:
extract_names(content_df["keywords"][0])

'jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life'

In [31]:
content_df["keywords"] = content_df["keywords"].apply(extract_names)
# content_df["cast"] = content_df["cast"].apply(extract_names)
# content_df["crew"] = content_df["crew"].apply(extract_names)


In [32]:
content_df

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de..."
...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,cia airport cat jew orderly airplane father-in...,"[{'cast_id': 1, 'character': 'Gaylord ""Greg"" F...","[{'credit_id': '52fe4303c3a36847f8033ca9', 'de..."
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,drug addiction junkie heroin speed diet unsoci...,"[{'cast_id': 29, 'character': 'Sara Goldfarb',...","[{'credit_id': '52fe4263c3a36847f801a72b', 'de..."
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,independent film awol fort polk louisiana targ...,"[{'cast_id': 1, 'character': 'Pvt. Roland Bozz...","[{'credit_id': '52fe43a39251416c75018317', 'de..."
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,,"[{'cast_id': 1, 'character': 'Buddy Visalo', '...","[{'credit_id': '52fe46c2c3a368484e0a2471', 'de..."


In [33]:
content_df["cast"][0]

"[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d', 'gender': 2, 'id': 7167, 'name': 'Don Rickles', 'order': 2, 'profile_path': '/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg'}, {'cast_id': 17, 'character': 'Slinky Dog (voice)', 'credit_id': '52fe4284c3a36847f8024fa1', 'gender': 2, 'id': 12899, 'name': 'Jim Varney', 'order': 3, 'profile_path': '/eIo2jVVXYgjDtaHoF19Ll9vtW7h.jpg'}, {'cast_id': 18, 'character': 'Rex (voice)', 'credit_id': '52fe4284c3a36847f8024fa5', 'gender': 2, 'id': 12900, 'name': 'Wallace Shawn', 'order': 4, 'profile_path': '/oGE6JqPP2xH4t

In [34]:
def convert_cast(text):
    return " ".join([d["name"] for d in ast.literal_eval(text)][:3])

In [35]:
content_df["cast"] = content_df["cast"].apply(convert_cast)

In [36]:
content_df.head()

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,"[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,"[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,"[{'credit_id': '52fe466a9251416c75077a89', 'de..."
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,"[{'credit_id': '52fe44779251416c91011acb', 'de..."
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,"[{'credit_id': '52fe44959251416c75039ed7', 'de..."


In [37]:
content_df["crew"][0]

'[{\'credit_id\': \'52fe4284c3a36847f8024f49\', \'department\': \'Directing\', \'gender\': 2, \'id\': 7879, \'job\': \'Director\', \'name\': \'John Lasseter\', \'profile_path\': \'/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f4f\', \'department\': \'Writing\', \'gender\': 2, \'id\': 12891, \'job\': \'Screenplay\', \'name\': \'Joss Whedon\', \'profile_path\': \'/dTiVsuaTVTeGmvkhcyJvKp2A5kr.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f55\', \'department\': \'Writing\', \'gender\': 2, \'id\': 7, \'job\': \'Screenplay\', \'name\': \'Andrew Stanton\', \'profile_path\': \'/pvQWsu0qc8JFQhMVJkTHuexUAa1.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f5b\', \'department\': \'Writing\', \'gender\': 2, \'id\': 12892, \'job\': \'Screenplay\', \'name\': \'Joel Cohen\', \'profile_path\': \'/dAubAiZcvKFbboWlj7oXOkZnTSu.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f61\', \'department\': \'Writing\', \'gender\': 0, \'id\': 12893, \'job\': \'Screenplay\', \'name\': \'A

In [38]:
def convert_crew(text):
    return " ".join([d["name"] for d in ast.literal_eval(text) if d['job']=="Director"])

In [39]:
content_df["crew"] = content_df["crew"].apply(convert_crew)

In [40]:
content_df.head()

,movieId,title,overview,genres,keywords,cast,crew
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,John Lasseter
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,Joe Johnston
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,Howard Deutch
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,Forest Whitaker
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,Charles Shyer


In [41]:
content_df["combined_features"] = (
    # content_df["overview"] + " " +
    (content_df["genres"] + " ") * 3 +
    content_df["keywords"] + " " +
    content_df["cast"]+" "+content_df["crew"]
)


In [42]:
# Clean genres
content_df["genres_clean"] = (
    content_df["genres"]
    .str.replace("|", " ", regex=False)
    .str.lower()
)

# Clean keywords (already space separated usually)
content_df["keywords_clean"] = (
    content_df["keywords"]
    .str.lower()
)

# Final explainable text
content_df["explainable_text"] = (
    content_df["genres_clean"] + " " +
    content_df["keywords_clean"]
)


In [43]:
content_df.head()

,movieId,title,overview,genres,keywords,cast,crew,combined_features,genres_clean,keywords_clean,explainable_text
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,John Lasseter,Animation Children's Comedy Animation Children...,animation children's comedy,jealousy toy boy friendship friends rivalry bo...,animation children's comedy jealousy toy boy f...
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,Joe Johnston,Adventure Children's Fantasy Adventure Childre...,adventure children's fantasy,board game disappearance based on children's b...,adventure children's fantasy board game disapp...
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,Howard Deutch,Comedy Romance Comedy Romance Comedy Romance f...,comedy romance,fishing best friend duringcreditsstinger old men,comedy romance fishing best friend duringcredi...
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,Forest Whitaker,Comedy Drama Comedy Drama Comedy Drama based o...,comedy drama,based on novel interracial relationship single...,comedy drama based on novel interracial relati...
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,Charles Shyer,Comedy Comedy Comedy baby midlife crisis confi...,comedy,baby midlife crisis confidence aging daughter ...,comedy baby midlife crisis confidence aging da...


In [44]:
# from sklearn.feature_extraction.text import TfidfVectorizer

# tfidf = TfidfVectorizer(
#     stop_words="english",
#     max_features=10000,
#     ngram_range=(1, 2)
# )

# tfidf_matrix = tfidf.fit_transform(content_df["combined_features"])
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_sim = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

tfidf_matrix_sim = tfidf_sim.fit_transform(
    content_df["combined_features"]
)


In [45]:
# tfidf_matrix

In [46]:
# from sklearn.metrics.pairwise import cosine_similarity

# cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


In [47]:
# movie_index = pd.Series(
#     content_df.index,
#     index=content_df["movieId"]
# )


In [48]:
cosine_sim = cosine_similarity(
    tfidf_matrix_sim,
    tfidf_matrix_sim
)


In [49]:
movie_index = pd.Series(
    content_df.index,
    index=content_df["movieId"]
)


In [50]:
def recommend_movies_content(movie_id, top_n=10):
    if movie_id not in movie_index:
        return "Movie not found"

    idx = movie_index[movie_id]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n + 1]

    movie_indices = [i[0] for i in similarity_scores]

    return content_df.iloc[movie_indices][["movieId", "title"]]


In [51]:
content_df[content_df["title"] == "Life (1999)"][["movieId", "title"]]


,movieId,title
2502,2587,Life (1999)


In [52]:
recommend_movies_content(movie_id=1	)


,movieId,title
3032,3114,Toy Story 2 (1999)
1055,1064,Aladdin and the King of Thieves (1996)
2061,2142,"American Tail: Fievel Goes West, An (1991)"
2060,2141,"American Tail, An (1986)"
2270,2355,"Bug's Life, A (1998)"
3524,3611,Saludos Amigos (1943)
589,596,Pinocchio (1940)
3664,3751,Chicken Run (2000)
2269,2354,"Rugrats Movie, The (1998)"
240,244,Gumby: The Movie (1995)


In [53]:
content_df.sample(10)

,movieId,title,overview,genres,keywords,cast,crew,combined_features,genres_clean,keywords_clean,explainable_text
1150,1169,American Dream (1990),Chronicles the six-month strike at Hormel in A...,Documentary,woman director,Jesse Jackson Juan Munoz Ray Rogers,Barbara Kopple Cathy Caplan Tom Haneke Lawrenc...,Documentary Documentary Documentary woman dire...,documentary,woman director,documentary woman director
2500,2585,"Lovers of the Arctic Circle, The (Los Amantes ...",Otto and Ana are kids when they meet each othe...,Drama Romance,suicide love at first sight brother sister rel...,Najwa Nimri Fele Martínez Joost Siedhoff,Julio Médem,Drama Romance Drama Romance Drama Romance suic...,drama romance,suicide love at first sight brother sister rel...,drama romance suicide love at first sight brot...
61,62,Mr. Holland's Opus (1995),"In 1965, passionate musician Glenn Holland tak...",Drama,composer mentor deaf-mute musical apprentice p...,Richard Dreyfuss Glenne Headly Jay Thomas,Stephen Herek,Drama Drama Drama composer mentor deaf-mute mu...,drama,composer mentor deaf-mute musical apprentice p...,drama composer mentor deaf-mute musical appren...
504,509,"Piano, The (1993)","After a long voyage from Scotland, pianist Ada...",Drama Romance,love triangle scotland mother adultery sexuali...,Holly Hunter Harvey Keitel Sam Neill,Jane Campion,Drama Romance Drama Romance Drama Romance love...,drama romance,love triangle scotland mother adultery sexuali...,drama romance love triangle scotland mother ad...
3076,3158,"Emperor and the Assassin, The (Jing ke ci qin ...","In pre-unified China, the King of Qin sends hi...",Drama,china assassin kingdom occupying power governa...,Gong Li Zhang Fengyi Sun Zhou,Chen Kaige,Drama Drama Drama china assassin kingdom occup...,drama,china assassin kingdom occupying power governa...,drama china assassin kingdom occupying power g...
2091,2172,"Strike! (a.k.a. All I Wanna Do, The Hairy Bird...",If there's one thing this wild group of friend...,Comedy,woman director,Kirsten Dunst Gaby Hoffmann Lynn Redgrave,Sarah Kernochan,Comedy Comedy Comedy woman director Kirsten Du...,comedy,woman director,comedy woman director
128,130,Angela (1995),A ten year old girl named Angela leads her six...,Drama,independent film woman director,Miranda Rhyne Charlotte Eve Blythe Anna Levine,Rebecca Miller,Drama Drama Drama independent film woman direc...,drama,independent film woman director,drama independent film woman director
1674,1728,"Winter Guest, The (1997)",The plot of the story is interactions between ...,Drama,independent film,Emma Thompson Phyllida Law Sheila Reid,Alan Rickman,Drama Drama Drama independent film Emma Thomps...,drama,independent film,drama independent film
3405,3491,My Chauffeur (1986),A feisty young woman accepts a mysterious offe...,Comedy,,Deborah Foreman Sam J. Jones Howard Hesseman,David Beaird,Comedy Comedy Comedy Deborah Foreman Sam J. J...,comedy,,comedy
274,278,Miami Rhapsody (1995),Gwyn Marcus has always wanted a marriage like ...,Comedy,,Sarah Jessica Parker Antonio Banderas Mia Farrow,David Frankel,Comedy Comedy Comedy Sarah Jessica Parker Ant...,comedy,,comedy


In [54]:
ratings_clean


,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [55]:
!pip install scikit-surprise

In [56]:
import surprise
print(surprise.__version__)


1.1.4


In [60]:
!pip install scikit-surprise


In [62]:
!pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.3
    Uninstalling numpy-2.4.3:
      Successfully uninstalled numpy-2.4.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is 

In [57]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split
import math


In [58]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings_clean[["userId", "movieId", "rating"]],
    reader
)


In [59]:
svd = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)


In [60]:
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

svd.fit(trainset)


In [61]:
def cf_score(user_id, movie_id):
    return svd.predict(user_id, movie_id).est


In [62]:
def rank_movies_cf(user_id, candidate_movie_ids, top_n=10):
    scores = []

    for movie_id in candidate_movie_ids:
        score = cf_score(user_id, movie_id)
        scores.append((movie_id, score))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_n]


In [63]:
predictions = svd.test(testset)




In [64]:
# Step 1: get candidates from content-based model
content_candidates = recommend_movies_content(movie_id=1, top_n=10)["movieId"]

# Step 2: rank them using CF
ranked = rank_movies_cf(user_id=10, candidate_movie_ids=content_candidates)

# Step 3: show ranked movies
ranked_movies = pd.DataFrame(ranked, columns=["movieId", "cf_score"])
ranked_movies.merge(
    content_df[["movieId", "title"]],
    on="movieId"
)


,movieId,cf_score,title
0,3751,4.867838,Chicken Run (2000)
1,3114,4.349145,Toy Story 2 (1999)
2,2141,4.223761,"American Tail, An (1986)"
3,2355,4.212605,"Bug's Life, A (1998)"
4,596,3.907944,Pinocchio (1940)
5,2142,3.892748,"American Tail: Fievel Goes West, An (1991)"
6,3611,3.662182,Saludos Amigos (1943)
7,1064,3.613853,Aladdin and the King of Thieves (1996)
8,244,3.471190,Gumby: The Movie (1995)
9,2354,3.335601,"Rugrats Movie, The (1998)"


In [65]:
!pip install shap


  Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 21.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


In [66]:
# import shap
# import numpy as np

# def similarity_model(X):
#     return cosine_similarity(X, reference_vector).ravel()


In [67]:
# explanations = {}

# for movie_id in content_candidates_df["movieId"]:
#     shap_df = explain_content_similarity(
#         seed_movie_id=1,
#         candidate_movie_id=movie_id,
#         top_k=5
#     )
#     explanations[movie_id] = shap_df["feature"].tolist()


In [68]:
content_df["genres_clean"] = (
    content_df["genres"]
    .str.replace("|", " ", regex=False)
    .str.lower()
)


In [69]:
def get_set(series_value):
    if isinstance(series_value, str):
        return set(series_value.lower().split())
    return set()


In [70]:
def generate_rich_explanation(seed_movie_id, candidate_movie_id):
    # Seed movie data
    seed_row = content_df[content_df["movieId"] == seed_movie_id].iloc[0]
    cand_row = content_df[content_df["movieId"] == candidate_movie_id].iloc[0]

    seed_title = seed_row["title"]

    # Genres
    seed_genres = get_set(seed_row["genres_clean"])
    cand_genres = get_set(cand_row["genres_clean"])
    shared_genres = seed_genres.intersection(cand_genres)

    # Cast
    seed_cast = get_set(seed_row["cast"])
    cand_cast = get_set(cand_row["cast"])
    shared_cast = seed_cast.intersection(cand_cast)

    # Director
    seed_director = seed_row.get("crew", "")
    cand_director = cand_row.get("crew", "")
    same_director = (
        isinstance(seed_director, str)
        and isinstance(cand_director, str)
        and seed_director.lower() == cand_director.lower()
        and seed_director != ""
    )

    reasons = []

    if shared_genres:
        reasons.append(
            "shared genres like " +
            ", ".join([g.title() for g in list(shared_genres)[:2]])
        )

    if shared_cast:
        reasons.append(
            "common cast members such as " +
            ", ".join([c.title() for c in list(shared_cast)[:2]])
        )

    if same_director:
        reasons.append(
            f"the same director ({seed_director})"
        )

    if not reasons:
        return (
            f"Recommended because it is similar in overall theme to "
            f"'{seed_title}'."
        )

    return (
        f"Recommended because it shares "
        + ", and ".join(reasons)
        + f" with '{seed_title}'."
    )


In [71]:
seed_movie_id = 2

content_candidates = recommend_movies_content(
    movie_id=seed_movie_id,
    top_n=10
)

explanations = {}

for movie_id in content_candidates["movieId"]:
    explanations[movie_id] = generate_rich_explanation(
        seed_movie_id,
        movie_id
    )


In [72]:
ranked = rank_movies_cf(
    user_id=10,
    candidate_movie_ids=content_candidates["movieId"].tolist()
)

ranked_df = (
    pd.DataFrame(ranked, columns=["movieId", "cf_score"])
    .merge(content_df[["movieId", "title"]], on="movieId")
)


In [73]:
ranked_df["explanation"] = ranked_df["movieId"].apply(
    lambda x: explanations[x]
)

ranked_df


,movieId,cf_score,title,explanation
0,2161,4.883018,"NeverEnding Story, The (1984)",Recommended because it shares shared genres li...
1,1967,4.874100,Labyrinth (1986),Recommended because it shares shared genres li...
2,2005,4.333994,"Goonies, The (1985)",Recommended because it shares shared genres li...
3,2043,4.225967,Darby O'Gill and the Little People (1959),Recommended because it shares shared genres li...
4,1009,4.118782,Escape to Witch Mountain (1975),Recommended because it shares shared genres li...
5,60,4.004665,"Indian in the Cupboard, The (1995)",Recommended because it shares shared genres li...
6,2399,3.678919,Santa Claus: The Movie (1985),Recommended because it shares shared genres li...
7,56,3.299471,Kids of the Round Table (1995),Recommended because it shares shared genres li...
8,2162,2.851041,"NeverEnding Story II: The Next Chapter, The (1...",Recommended because it shares shared genres li...
9,126,2.727981,"NeverEnding Story III, The (1994)",Recommended because it shares shared genres li...


In [74]:
ranked_df["explanation"][8]

"Recommended because it shares shared genres like Children'S, Fantasy, and common cast members such as Jonathan with 'Jumanji (1995)'."

In [75]:
content_df

,movieId,title,overview,genres,keywords,cast,crew,combined_features,genres_clean,keywords_clean,explainable_text
0,1,Toy Story (1995),"Led by Woody, Andy's toys live happily in his ...",Animation Children's Comedy,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles,John Lasseter,Animation Children's Comedy Animation Children...,animation children's comedy,jealousy toy boy friendship friends rivalry bo...,animation children's comedy jealousy toy boy f...
1,2,Jumanji (1995),When siblings Judy and Peter discover an encha...,Adventure Children's Fantasy,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst,Joe Johnston,Adventure Children's Fantasy Adventure Childre...,adventure children's fantasy,board game disappearance based on children's b...,adventure children's fantasy board game disapp...
2,3,Grumpier Old Men (1995),A family wedding reignites the ancient feud be...,Comedy Romance,fishing best friend duringcreditsstinger old men,Walter Matthau Jack Lemmon Ann-Margret,Howard Deutch,Comedy Romance Comedy Romance Comedy Romance f...,comedy romance,fishing best friend duringcreditsstinger old men,comedy romance fishing best friend duringcredi...
3,4,Waiting to Exhale (1995),"Cheated on, mistreated and stepped on, the wom...",Comedy Drama,based on novel interracial relationship single...,Whitney Houston Angela Bassett Loretta Devine,Forest Whitaker,Comedy Drama Comedy Drama Comedy Drama based o...,comedy drama,based on novel interracial relationship single...,comedy drama based on novel interracial relati...
4,5,Father of the Bride Part II (1995),Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,Steve Martin Diane Keaton Martin Short,Charles Shyer,Comedy Comedy Comedy baby midlife crisis confi...,comedy,baby midlife crisis confidence aging daughter ...,comedy baby midlife crisis confidence aging da...
...,...,...,...,...,...,...,...,...,...,...,...
3858,3948,Meet the Parents (2000),"Greg Focker is ready to marry his girlfriend, ...",Comedy,cia airport cat jew orderly airplane father-in...,Ben Stiller Robert De Niro Teri Polo,Jay Roach,Comedy Comedy Comedy cia airport cat jew order...,comedy,cia airport cat jew orderly airplane father-in...,comedy cia airport cat jew orderly airplane fa...
3859,3949,Requiem for a Dream (2000),The hopes and dreams of four ambitious people ...,Drama,drug addiction junkie heroin speed diet unsoci...,Ellen Burstyn Jared Leto Jennifer Connelly,Darren Aronofsky,Drama Drama Drama drug addiction junkie heroin...,drama,drug addiction junkie heroin speed diet unsoci...,drama drug addiction junkie heroin speed diet ...
3860,3950,Tigerland (2000),A group of recruits go through Advanced Infant...,Drama,independent film awol fort polk louisiana targ...,Colin Farrell Matthew Davis Clifton Collins Jr,Joel Schumacher,Drama Drama Drama independent film awol fort p...,drama,independent film awol fort polk louisiana targ...,drama independent film awol fort polk louisian...
3861,3951,Two Family House (2000),Buddy Visalo (Michael Rispoli) is a factory wo...,Drama,,Michael Rispoli Kelly Macdonald Kathrine Narducci,Raymond De Felitta,Drama Drama Drama Michael Rispoli Kelly Macdo...,drama,,drama


In [76]:
rmse = accuracy.rmse(predictions)

RMSE: 0.8737


In [77]:
def get_relevant_items(testset, threshold=4.0):
    relevant = {}

    for uid, iid, true_r in testset:
        if true_r >= threshold:
            relevant.setdefault(uid, set()).add(iid)

    return relevant

relevant_items = get_relevant_items(testset)




In [78]:
def get_top_k_predictions(model, testset, k=10):
    user_predictions = {}

    for uid, iid, true_r in testset:
        est = model.predict(uid, iid).est
        user_predictions.setdefault(uid, []).append((iid, est))

    # Sort predictions for each user
    for uid in user_predictions:
        user_predictions[uid].sort(key=lambda x: x[1], reverse=True)
        user_predictions[uid] = user_predictions[uid][:k]

    return user_predictions

top_k_preds = get_top_k_predictions(svd, testset, k=10)


In [79]:
def precision_recall_at_k(relevant, predicted, k=10):
    precisions = []
    recalls = []

    for uid in relevant:
        if uid not in predicted:
            continue

        pred_items = {iid for iid, _ in predicted[uid]}
        rel_items = relevant[uid]

        tp = len(pred_items & rel_items)

        precisions.append(tp / k)
        recalls.append(tp / len(rel_items))

    return (
        sum(precisions) / len(precisions),
        sum(recalls) / len(recalls)
    )

precision, recall = precision_recall_at_k(
    relevant_items,
    top_k_preds,
    k=10
)

print("Precision@10:", precision)
print("Recall@10:", recall)


Precision@10: 0.6850425212606303
Recall@10: 0.641410068125028


In [80]:
def ndcg_at_k(relevant, predicted, k=10):
    ndcgs = []

    for uid in relevant:
        if uid not in predicted:
            continue

        dcg = 0.0
        for rank, (iid, _) in enumerate(predicted[uid], start=1):
            if iid in relevant[uid]:
                dcg += 1 / math.log2(rank + 1)

        ideal_dcg = sum(
            1 / math.log2(rank + 1)
            for rank in range(1, min(len(relevant[uid]), k) + 1)
        )

        if ideal_dcg > 0:
            ndcgs.append(dcg / ideal_dcg)

    return sum(ndcgs) / len(ndcgs)

ndcg = ndcg_at_k(
    relevant_items,
    top_k_preds,
    k=10
)

print("NDCG@10:", ndcg)


NDCG@10: 0.873798409954208
